# PIN Blocks and MACs

This notebook covers the two security fields most test harnesses need:

- **Field 52**: the encrypted PIN block (ISO 9564-1)
- **Fields 64 and 128**: the message authentication code (ISO 9797-1)

> **Test and simulation use only.** Real systems keep clear PINs and keys inside an HSM. Everything below uses published test keys.

Requires the security extra: `pip install iso8583sim[security]`

In [1]:
import sys

sys.path.insert(0, "..")  # For development - remove in production

In [2]:
from iso8583sim.core.builder import ISO8583Builder
from iso8583sim.core.samples import sample_message
from iso8583sim.core.validator import ISO8583Validator
from iso8583sim.security import (
    SecurityError,
    decrypt_pin_block,
    encode_pin_block,
    encrypt_pin_block,
    generate_mac,
)

PAN = "4111111111111111"
TDES_KEY = "0123456789ABCDEFFEDCBA9876543210"  # double length TDES test key
AES_KEY = "00112233445566778899AABBCCDDEEFF"  # AES-128 test key

## How a format 0 PIN block is built

Format 0 XORs two 8-byte fields:

- the **PIN field**: `0`, the PIN length, the PIN, then `F` padding
- the **PAN field**: `0000`, then the 12 rightmost PAN digits excluding the check digit

In [3]:
pin_field = "04" + "1234" + "F" * 10
pan_field = "0000" + PAN[:-1][-12:]
clear_block = encode_pin_block("1234", PAN, fmt=0).hex().upper()

print(f"PIN field  {pin_field}")
print(f"PAN field  {pan_field}")
print(f"XOR        {clear_block}")

PIN field  041234FFFFFFFFFF
PAN field  0000111111111111
XOR        041225EEEEEEEEEE


## Encrypting and decrypting

Formats 0, 1 and 3 use TDES and fit field 52 in every version (8 bytes).

In [4]:
for fmt in (0, 1, 3):
    pan = None if fmt == 1 else PAN  # format 1 doesn't use the PAN
    block = encrypt_pin_block("1234", pan, TDES_KEY, fmt=fmt)
    print(f"format {fmt}: {block}  ->  PIN {decrypt_pin_block(block, pan, TDES_KEY, fmt=fmt)}")

format 0: 2A3D408A1977DDE9  ->  PIN 1234
format 1: 32CA5E9CFACCDD13  ->  PIN 1234
format 3: DB0232CBF09417A3  ->  PIN 1234


Formats 1 and 3 use random fill, so the same PIN gives a different block every time. Format 0 is deterministic:

In [5]:
print(encrypt_pin_block("1234", PAN, TDES_KEY, fmt=3))
print(encrypt_pin_block("1234", PAN, TDES_KEY, fmt=3))
print(encrypt_pin_block("1234", PAN, TDES_KEY, fmt=0))
print(encrypt_pin_block("1234", PAN, TDES_KEY, fmt=0))

F6395E105E4C078C
09F271898E92E412
2A3D408A1977DDE9
2A3D408A1977DDE9


Format 4 uses AES and produces 16 bytes, which fits field 52 in the 1993 and 2003 versions:

In [6]:
block4 = encrypt_pin_block("1234", PAN, AES_KEY, fmt=4)
print(block4, f"({len(block4) // 2} bytes)")
print("PIN:", decrypt_pin_block(block4, PAN, AES_KEY, fmt=4))

8C8C8DEC023C741FAFD3985C3F7C65DA (16 bytes)
PIN: 1234


A wrong key or PAN doesn't decrypt to a valid PIN block, so the mistake is caught instead of returning a wrong PIN:

In [7]:
block = encrypt_pin_block("1234", PAN, TDES_KEY, fmt=0)
try:
    decrypt_pin_block(block, "5555555555554444", TDES_KEY, fmt=0)
except SecurityError as e:
    print("SecurityError:", e)

SecurityError: Format 0 PIN block has invalid fill (wrong key, PAN or format?)


## MACs

The default is the ANSI X9.19 retail MAC (ISO 9797-1 algorithm 3). `build_with_mac` builds the message and puts the MAC in its last field: 64, or 128 when there is a secondary bitmap.

In [8]:
message = sample_message(pan=PAN)
message.fields[52] = encrypt_pin_block("1234", PAN, TDES_KEY)

raw = ISO8583Builder().build_with_mac(message, TDES_KEY)
print(raw)
print("MAC (field 64):", message.fields[64])
print("Valid:", ISO8583Validator().verify_mac(raw, TDES_KEY))

01007024058000C09001164111111111111111000000000000001000123456261205100100TERM0001MERCHANT123456 8402A3D408A1977DDE98D45A2D4959BC19E
MAC (field 64): 8D45A2D4959BC19E
Valid: True


Any change to the message breaks the MAC:

In [9]:
tampered = raw.replace("000000001000", "000000009000")  # 10.00 -> 90.00
print("Tampered message valid:", ISO8583Validator().verify_mac(tampered, TDES_KEY))
print("Wrong key valid:       ", ISO8583Validator().verify_mac(raw, "11" * 16))

Tampered message valid: False
Wrong key valid:        False


You can also MAC arbitrary data, with algorithm 1 (CBC-MAC) or 3, padding method 1 or 2, and a shorter length:

In [10]:
data = b"any bytes you need to authenticate"
print("retail MAC:     ", generate_mac(data, TDES_KEY))
print("CBC-MAC:        ", generate_mac(data, TDES_KEY, algorithm=1))
print("4-byte, pad 2:  ", generate_mac(data, TDES_KEY, padding=2, length=4))

retail MAC:      B73652F8A6BD991A
CBC-MAC:         84B3A77546B5CEF7
4-byte, pad 2:   D3C7AB6D


Next: [10_wire_formats.ipynb](10_wire_formats.ipynb) sends these messages as real network bytes.